In [0]:
import dlt
from pyspark.sql.functions import *

In [0]:
@dlt.table(
  comment="Raw student data ingestion"
)
def bronze_students():
    return (
        spark.read.format("csv")
        .option("header","true")
        .load("/Volumes/workspace/default/my_volume_2/decp_students.csv")
    )

In [0]:
@dlt.table(
  comment="Clean student data"
)
@dlt.expect("valid_marks","marks > 0")
def silver_students():
    return (
        dlt.read("bronze_students")
        .withColumn("marks", col("marks").cast("int"))
        .filter("marks > 0")
    )

In [0]:
@dlt.table(
  comment="Average marks per department"
)
def gold_department_avg():
    return (
        dlt.read("silver_students")
        .groupBy("department")
        .agg(avg("marks").alias("avg_marks"))
    )